# Explore performance evaluation summary results

In [61]:
import json
from pathlib import Path
import pandas as pd

LOGS_DIR = Path("../logs")

# ------------------------------------------------------------------
# Helper: conditional IPSNR (mean over samples with PP IoU > 0, plus n)
# ------------------------------------------------------------------
def conditional_ipsnr(csv_path, ipsnr_col="pp_ipsnr", iou_col="pp_iou"):
    if not csv_path.exists():
        return "N/A"
    df = pd.read_csv(csv_path)
    if ipsnr_col not in df.columns or iou_col not in df.columns:
        return "N/A"
    subset = df.loc[df[iou_col] > 0, ipsnr_col]
    if len(subset) == 0:
        return "N/A"
    return f"{subset.mean():.2f} ({len(subset)})"

# ------------------------------------------------------------------
# Main loop: build model rows + baseline rows
# ------------------------------------------------------------------
COLUMN_ORDER = ["auprc", "precision (Post-PP)", "recall (Post-PP)", "boundary_f1 (Post-PP)", "ipsnr"]

json_files = sorted(list(LOGS_DIR.glob("e1_*best_val_auprc*.json")))
model_rows = []
baseline_all_zero = {}
baseline_prevalence = {}

for file in json_files:
    with open(file, "r") as f:
        content = json.load(f)
    exp_name = content.get("experiment", "")
    if not exp_name.startswith("e1"):
        continue
    metrics = content.get("metrics", {})

    per_sample_csv = LOGS_DIR / file.name.replace("_test_summary.json", "_test_per_sample.csv")

    row = {
        "experiment": exp_name,
        "auprc": metrics.get("auprc", {}).get("mean"),
        "precision (Post-PP)": metrics.get("pp_precision", {}).get("mean"),
        "recall (Post-PP)": metrics.get("pp_recall", {}).get("mean"),
        "boundary_f1 (Post-PP)": metrics.get("pp_boundary_f1", {}).get("mean"),
        "ipsnr": conditional_ipsnr(per_sample_csv),
    }
    model_rows.append(row)

    if not baseline_all_zero:
        b_zero = content.get("baseline_all_zero", {})
        baseline_all_zero = {
            "auprc": b_zero.get("auprc", {}).get("mean"),
            "precision (Post-PP)": b_zero.get("precision", {}).get("mean"),
            "recall (Post-PP)": b_zero.get("recall", {}).get("mean"),
            "boundary_f1 (Post-PP)": b_zero.get("pp_boundary_f1", {}).get("mean"),  # not computed -> None -> N/A
            "ipsnr": None,
        }
    if not baseline_prevalence:
        b_prev = content.get("baseline_prevalence_rate", {})
        baseline_prevalence = {
            "auprc": b_prev.get("auprc", {}).get("mean"),
            "precision (Post-PP)": b_prev.get("precision", {}).get("mean"),
            "recall (Post-PP)": b_prev.get("recall", {}).get("mean"),
            "boundary_f1 (Post-PP)": b_prev.get("pp_boundary_f1", {}).get("mean"),
            "ipsnr": None,
        }

# Build Model DataFrame
summary_df = pd.DataFrame(model_rows).set_index("experiment").reindex(columns=COLUMN_ORDER)

# Build Baseline DataFrame
baseline_df = pd.DataFrame(
    [baseline_all_zero, baseline_prevalence],
    index=["Baseline: All-Zero", "Baseline: Prevalence-Rate"]
).reindex(columns=COLUMN_ORDER)

# Combine into final table
final_table = pd.concat([summary_df, baseline_df])

# Explicit display order, keyed by experiment filename (matches EXPERIMENT_LABELS keys)
EXPERIMENT_ORDER = [
    "e1_haunet_direct_focal_dice",                # E1
    "e1_haunet_direct_focal_dice_no_cov_encoder",  # E1a
    "e1_haunet_direct_focal_dice_no_dem",          # E1b
    "e1_haunet_direct_focal_dice_no_landcover",    # E1c
    "e1_haunet_direct_focal_dice_dilated",         # E1d
]

# Reorder final_table's model rows to match EXPERIMENT_ORDER, keep baseline rows after
ordered_index = [e for e in EXPERIMENT_ORDER if e in final_table.index] + \
                [b for b in ["Baseline: All-Zero", "Baseline: Prevalence-Rate"] if b in final_table.index]
final_table = final_table.reindex(ordered_index)

In [62]:
# ------------------------------------------------------------------
# Notebook display (numeric columns formatted, ipsnr passed through)
# ------------------------------------------------------------------
numeric_cols = ["auprc", "precision (Post-PP)", "recall (Post-PP)", "boundary_f1 (Post-PP)"]

print("--- Final Model Evaluation vs Fixed Baselines (Post-PP Precision/Recall) ---")
display(
    final_table.style
    .highlight_max(subset=pd.IndexSlice[summary_df.index, numeric_cols], axis=0, color="lightgreen")
    .format({col: "{:.6f}" for col in numeric_cols}, na_rep="N/A")
    .format({"ipsnr": lambda x: "N/A" if pd.isna(x) else str(x)})
)

# ------------------------------------------------------------------
# Human-readable labels for report display
# ------------------------------------------------------------------
COLUMN_LABELS = {
    "auprc": "AUPRC",
    "precision (Post-PP)": "Precision",
    "recall (Post-PP)": "Recall",
    "boundary_f1 (Post-PP)": "Boundary F1",
    "ipsnr": "IPSNR (n)",
}

EXPERIMENT_LABELS = {
    # --- E1: HAUNet, direct pipeline ---
    "e1_haunet_direct_focal_dice":               "E1: Full (HAUNet)",
    "e1_haunet_direct_focal_dice_no_cov_encoder": "E1a: No Cov.\\ Pyramid Encoder",
    "e1_haunet_direct_focal_dice_no_dem":         "E1b: No DEM",
    "e1_haunet_direct_focal_dice_no_landcover":   "E1c: No Land Cover",
    "e1_haunet_direct_focal_dice_dilated":        "E1d: Dilated Target",

    # --- E2: Autoencoder baseline, direct pipeline ---
    # "e2_autoencoder_direct_focal_dice":         "E2: Autoencoder (Direct)",

    # --- E3: HAUNet, cascaded Stage 1 ---
    # "e3_haunet_cascaded_stage1_...":            "E3: HAUNet (Cascaded, S1)",

    # --- E4: HAUNet, cascaded Stage 2 ---
    # "e4_haunet_cascaded_stage2_...":            "E4: HAUNet (Cascaded, S2)",

    # --- E5: Autoencoder baseline, cascaded ---
    # "e5_autoencoder_cascaded_...":              "E5: Autoencoder (Cascaded)",
}

BASELINE_LABELS = {
    "Baseline: All-Zero": "Baseline: All-Zero",
    "Baseline: Prevalence-Rate": "Baseline: Prevalence-Rate",
}

--- Final Model Evaluation vs Fixed Baselines (Post-PP Precision/Recall) ---


,auprc,precision (Post-PP),recall (Post-PP),boundary_f1 (Post-PP),ipsnr
e1_haunet_direct_focal_dice,0.010301,0.002129,0.340311,0.003033,0.02 (59)
e1_haunet_direct_focal_dice_no_cov_encoder,0.009764,0.004265,0.460508,0.003855,0.03 (73)
e1_haunet_direct_focal_dice_no_dem,0.004990,0.002046,0.310042,0.003337,0.02 (56)
e1_haunet_direct_focal_dice_no_landcover,0.005364,0.002577,0.505096,0.001960,0.02 (77)
e1_haunet_direct_focal_dice_dilated,0.016267,0.007190,0.310138,0.004290,0.07 (55)
Baseline: All-Zero,0.000618,0.000000,0.000000,None,N/A
Baseline: Prevalence-Rate,0.000618,0.000000,0.000000,None,N/A


In [63]:
# ------------------------------------------------------------------
# Bold the best (max) value per numeric column, ignoring baseline rows
# and skipping ipsnr (not a fair "higher is better" comparison given
# varying n)
# ------------------------------------------------------------------
formatted_df = final_table.copy().astype(object)

for col in formatted_df.columns:
    if col == "ipsnr":
        # Pass through as-is (already a formatted string or None)
        for idx in formatted_df.index:
            val = final_table.at[idx, col]
            formatted_df.at[idx, col] = "N/A" if pd.isna(val) else str(val)
        continue

    model_vals = final_table.loc[summary_df.index, col]
    if model_vals.isna().all():
        continue
    best_val = model_vals.max()
    for idx in formatted_df.index:
        val = final_table.at[idx, col]
        if pd.isna(val):
            formatted_df.at[idx, col] = "N/A"
        elif idx in summary_df.index and val == best_val:
            formatted_df.at[idx, col] = f"\\textbf{{{val:.4f}}}"
        else:
            formatted_df.at[idx, col] = f"{val:.4f}"

# ------------------------------------------------------------------
# Apply readable labels
# ------------------------------------------------------------------
formatted_df.index = [
    EXPERIMENT_LABELS.get(idx, BASELINE_LABELS.get(idx, str(idx).replace("_", r"\_")))
    for idx in formatted_df.index
]
formatted_df.columns = [COLUMN_LABELS.get(col, str(col)) for col in formatted_df.columns]

In [64]:
# ------------------------------------------------------------------
# Export to LaTeX, with a \midrule separating model rows from baselines
# ------------------------------------------------------------------
PROJECT_ROOT = Path("..").resolve()
output_dir = PROJECT_ROOT / "deliverables" / "tables"
output_dir.mkdir(parents=True, exist_ok=True)
csv_path = output_dir / "evaluation_results.csv"
tex_path = output_dir / "evaluation_table.tex"

final_table.to_csv(csv_path)

latex_tabular = formatted_df.to_latex(
    column_format="lccccc",
    escape=False,
)

lines = latex_tabular.splitlines()
baseline_row_start = None
for i, line in enumerate(lines):
    if line.strip().startswith("Baseline"):
        baseline_row_start = i
        break
if baseline_row_start is not None:
    lines.insert(baseline_row_start, "\\midrule")
latex_tabular = "\n".join(lines)

full_latex_table = f"""\\begin{{table}}[htbp]
\\centering
\\small
\\caption{{Model performance evaluation across feature ablations vs.\\ fixed baselines. Precision, Recall, Boundary F1, and IPSNR are computed on Otsu-thresholded predictions. IPSNR is reported as the mean over samples with nonzero post-processed IoU, with the contributing sample count in parentheses.}}
\\label{{tab:evaluation_results}}
{latex_tabular}
\\end{{table}}
"""

with open(tex_path, "w") as f:
    f.write(full_latex_table)

print(f"Successfully generated updated LaTeX snippet at: {tex_path}")

Successfully generated updated LaTeX snippet at: /home/jupyter/irp-th25/deliverables/tables/evaluation_table.tex


### Check ipsnr

In [60]:
import pandas as pd
from pathlib import Path

LOGS_DIR = Path("../logs")

configs = {
    "Full": "e1_haunet_direct_focal_dice_best_val_auprc_test_per_sample.csv",
    "Dilated": "e1_haunet_direct_focal_dice_dilated_best_val_auprc_test_per_sample.csv",
    "No Cov Pyramid": "e1_haunet_direct_focal_dice_no_cov_encoder_best_val_auprc_test_per_sample.csv",
    "No DEM": "e1_haunet_direct_focal_dice_no_dem_best_val_auprc_test_per_sample.csv",
    "No Land Cover": "e1_haunet_direct_focal_dice_no_landcover_best_val_auprc_test_per_sample.csv",
}

for name, fname in configs.items():
    path = LOGS_DIR / fname
    if not path.exists():
        print(f"[skip] {name}: {fname} not found")
        continue
    df = pd.read_csv(path)

    n_total = len(df)
    n_nonzero_ipsnr = df["ipsnr"].notna().sum()   # non-NaN = had a valid intersection
    n_nonzero_iou = (df["iou"] > 0).sum()          # samples with any true positive overlap at all

    print(f"\n--- {name} ---")
    print(f"  n_samples: {n_total}")
    print(f"  n_samples with valid (non-NaN) ipsnr: {n_nonzero_ipsnr}")
    print(f"  n_samples with iou > 0: {n_nonzero_iou}")
    print(f"  ipsnr mean (all valid): {df['ipsnr'].mean():.4f}")
    print(f"  ipsnr mean (only iou>0 samples): {df.loc[df['iou'] > 0, 'ipsnr'].mean():.4f}")
    print(f"  ipsnr median: {df['ipsnr'].median():.4f}")
    print(f"  ipsnr max: {df['ipsnr'].max():.4f}  (sample_id: "
          f"{df.loc[df['ipsnr'].idxmax(), 'sample_id'] if n_nonzero_ipsnr > 0 else 'N/A'})")
    print(f"  top 5 ipsnr values: {df['ipsnr'].nlargest(5).tolist()}")


--- Full ---
  n_samples: 121
  n_samples with valid (non-NaN) ipsnr: 121
  n_samples with iou > 0: 2
  ipsnr mean (all valid): 0.3618
  ipsnr mean (only iou>0 samples): 1.9527
  ipsnr median: 0.0442
  ipsnr max: 4.2790  (sample_id: LC08_L1TP_008052_20200813_20200813_01_RT_intersection_p00394)
  top 5 ipsnr values: [4.279021, 3.692351, 3.600191, 3.150324, 2.388698]

--- Dilated ---
  n_samples: 121
  n_samples with valid (non-NaN) ipsnr: 121
  n_samples with iou > 0: 5
  ipsnr mean (all valid): 0.3997
  ipsnr mean (only iou>0 samples): 2.7408
  ipsnr median: 0.0062
  ipsnr max: 4.6248  (sample_id: LC08_L1TP_033042_20200812_20200812_01_RT_intersection_p00174)
  top 5 ipsnr values: [4.624831, 4.547169, 4.480926, 4.271442, 4.258946]

--- No Cov Pyramid ---
  n_samples: 121
  n_samples with valid (non-NaN) ipsnr: 121
  n_samples with iou > 0: 4
  ipsnr mean (all valid): 0.4840
  ipsnr mean (only iou>0 samples): 3.2050
  ipsnr median: 0.0381
  ipsnr max: 5.2545  (sample_id: LC08_L1TP_22008

This means that ipsnr is very sensitive to the frequency of which the model predictions overlap with the targets.